# c15-2026 — Congestión Fiscal — Pipeline Colab

Pipeline modular para `c15-2026` en Google Colab. Ejecuta `src/stages` registrados en `src/stages/registry.py` vía `main.py --stage <name>`.
Datos en Drive (`DATA_DIR`/`OUTPUT_DIR` vía `os.getenv`, mapeados a `GDRIVE_FOLDER_ID`), manifests en `outputs/manifest_index.json` (CSV+JSON, sin XLSX).
Orden: `consolidation` → `preprocessing` → `feature_engineering` → `feature_selection` → `split` → `train` (validación temporal walk-forward, `main.py` respeta `Stage.is_done()`).


In [ ]:
# Cell 1: Mount Drive, configure env vars (AGENTS.md:5), load code from Drive (no git)
from google.colab import drive
import os, pathlib, shutil

# 1. Mount Google Drive (only mount call in notebook)
drive.mount('/content/drive')

# Project root on Drive (maps to GDRIVE_FOLDER_ID env)
PROJECT_DIR = "/content/drive/MyDrive/research/congestion_fiscal"
os.environ["DATA_DIR"] = f"{PROJECT_DIR}/data"
os.environ["OUTPUT_DIR"] = f"{PROJECT_DIR}/outputs"
os.environ["GDRIVE_NOTEBOOK_NAME"] = os.getenv("GDRIVE_NOTEBOOK_NAME", "experiments.ipynb")

# 2. Copy runtime code snapshot uploaded from local via MCP gdrive (Drive/code/ -> /content)
CODE_DIR = pathlib.Path(f"{PROJECT_DIR}/code")
REPO_DIR = pathlib.Path("/content/experiments")
if not CODE_DIR.exists():
    raise RuntimeError(f"Drive code snapshot missing: {CODE_DIR} — run /exp sync locally first (agent uploads code/ via MCP gdrive uploadFile)")
shutil.copytree(CODE_DIR, REPO_DIR, dirs_exist_ok=True)

%cd /content/experiments
print(f"[ok] DATA_DIR={os.environ['DATA_DIR']}")
print(f"[ok] OUTPUT_DIR={os.environ['OUTPUT_DIR']}")
print(f"[ok] GDRIVE_NOTEBOOK_NAME={os.environ['GDRIVE_NOTEBOOK_NAME']}")
print(f"[ok] code snapshot copied from {CODE_DIR}")
# Freshness proof: confirm you run the latest synced code (not a stale copy)
import datetime, yaml
_tp = REPO_DIR / "src/stages/train.py"
_st = _tp.stat()
print(f"[ok] train.py mtime={datetime.datetime.fromtimestamp(_st.st_mtime).strftime('%Y-%m-%d %H:%M:%S')} size={_st.st_size}")
_cfg = yaml.safe_load(open(REPO_DIR / "config.yaml", encoding="utf-8"))
print(f"[ok] training={_cfg.get('training', {}).get('active_models')} trials={( _cfg.get('models', {}).get('lightgbm', {}) or {}).get('trials')}")


In [ ]:
# Instalar dependencias (requirements.txt usa pandas, sklearn, pyyaml, optuna, xgboost, lightgbm, catboost)
!pip install -r requirements.txt --quiet
print("[ok] dependencias instaladas")


In [ ]:
# Verificar env vars y estructura Drive (OUTPUT_DIR/DATA_DIR vía os.getenv, no rutas absolutas)
import os
from pathlib import Path
from dotenv import load_dotenv

# Carga .env si existe (local), Colab override ya está en os.environ
# Colab env comes from the mount cell above, never from .env files (local-only)
# (no load_dotenv in Colab)
DATA_DIR = os.getenv("DATA_DIR", "../data")
OUTPUT_DIR = os.getenv("OUTPUT_DIR", "../outputs")
GDRIVE_NOTEBOOK_NAME = os.getenv("GDRIVE_NOTEBOOK_NAME", "experiments.ipynb")
print(f"DATA_DIR={DATA_DIR}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")
print(f"GDRIVE_NOTEBOOK_NAME={GDRIVE_NOTEBOOK_NAME}")

# Crear estructura Drive ordenada (mirror local ../outputs -> Drive outputs/*)
for d in [DATA_DIR, OUTPUT_DIR, f"{OUTPUT_DIR}/tables", f"{OUTPUT_DIR}/figures", f"{OUTPUT_DIR}/models", f"{OUTPUT_DIR}/predictions", f"{OUTPUT_DIR}/checkpoints", f"{OUTPUT_DIR}/logs"]:
    Path(d).mkdir(parents=True, exist_ok=True)
    print(f"[ok] {d}")

# Verificar registry (source of truth)
import src.stages  # noqa: F401 autoregistro
from src.stages.registry import get_registry
print(f"stages registrados: {list(get_registry().keys())}")
import yaml
cfg = yaml.safe_load(open("config.yaml", encoding="utf-8"))
print(f"models config: {list(cfg.get('models', {}).keys())}")
print(f"split: {cfg.get('split', {})}")


In [ ]:
# Stage: consolidation — 3.1 Data Consolidation (checks outputs/manifest_index.json#consolidation)
# Usa Stage.is_done() para skip si ya completado; Stage.run() publica manifest a manifest_index.json
!python main.py --stage consolidation  # writes outputs/manifest_index.json#consolidation.manifest
# Re-run (ignores is_done, no cascade): !python main.py --stage consolidation --force
# Manifest: outputs/manifest_index.json (CSV+JSON, no XLSX) + tables/*.csv


In [ ]:
# Stage: feature_engineering — 3.3 Feature Engineering (checks outputs/manifest_index.json#feature_engineering)
# Usa Stage.is_done() para skip si ya completado; Stage.run() publica manifest a manifest_index.json
!python main.py --stage feature_engineering  # writes outputs/manifest_index.json#feature_engineering.manifest
# Re-run (ignores is_done, no cascade): !python main.py --stage feature_engineering --force
# Manifest: outputs/manifest_index.json (CSV+JSON, no XLSX) + tables/*.csv


In [ ]:
# Stage: preprocessing — 3.2 Preprocessing (checks outputs/manifest_index.json#preprocessing)
# Usa Stage.is_done() para skip si ya completado; Stage.run() publica manifest a manifest_index.json
!python main.py --stage preprocessing  # writes outputs/manifest_index.json#preprocessing.manifest
# Re-run (ignores is_done, no cascade): !python main.py --stage preprocessing --force
# Manifest: outputs/manifest_index.json (CSV+JSON, no XLSX) + tables/*.csv


In [ ]:
# Stage: feature_selection — 3.4 Feature Selection (checks outputs/manifest_index.json#feature_selection)
# Usa Stage.is_done() para skip si ya completado; Stage.run() publica manifest a manifest_index.json
!python main.py --stage feature_selection  # writes outputs/manifest_index.json#feature_selection.manifest
# Re-run (ignores is_done, no cascade): !python main.py --stage feature_selection --force
# Manifest: outputs/manifest_index.json (CSV+JSON, no XLSX) + tables/*.csv


In [ ]:
# Stage: split — 3.5 Temporal Split (checks outputs/manifest_index.json#split)
# Usa Stage.is_done() para skip si ya completado; Stage.run() publica manifest a manifest_index.json
!python main.py --stage split  # writes outputs/manifest_index.json#split.manifest
# Re-run (ignores is_done, no cascade): !python main.py --stage split --force
# Manifest: outputs/manifest_index.json (CSV+JSON, no XLSX) + tables/*.csv


In [ ]:
# Pre-flight for the scale-up run (100 trials, full active set): versions + disk
import importlib, os, shutil
OUTPUT_DIR = os.getenv("OUTPUT_DIR", "../outputs")
for lib in ["sklearn", "pandas", "numpy", "optuna", "lightgbm", "xgboost", "catboost"]:
    try:
        m = importlib.import_module(lib)
        print(f"[ok] {lib} {getattr(m, "__version__", "installed")}")
    except ImportError:
        print(f"[MISSING] {lib} — run the pip install cell first")
free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
print(f"[ok] OUTPUT_DIR={OUTPUT_DIR} free={free_gb:.1f} GB")


In [ ]:
# Train progress status (read-only, no execution): per-model manifests + trial histories
import os, json, glob
from pathlib import Path
import yaml
OUTPUT_DIR = os.getenv("OUTPUT_DIR", "../outputs")
models_dir = Path(OUTPUT_DIR) / "models"
cfg = yaml.safe_load(open("config.yaml", encoding="utf-8"))
active = ((cfg.get("training") or {}).get("active_models") or [])
trials_cfg = {m: ((cfg.get("models") or {}).get(m) or {}).get("trials", "-") for m in active}
print(f"{"model":<16}{"status":<10}{"trials":<12}{"test f1":<10}{"tau"}")
for m in active:
    mp = models_dir / f"train_{m}_manifest.json"
    hp = models_dir / f"train_{m}_trials.json"
    if mp.exists():
        d = json.loads(mp.read_text(encoding="utf-8"))
        s = d.get("summary", {})
        print(f"{m:<16}{"done":<10}{str(trials_cfg[m]):<12}{round(s.get("f1_test", 0.0), 4):<10}{s.get("tau_star", "-")}")
    elif hp.exists():
        h = json.loads(hp.read_text(encoding="utf-8"))
        print(f"{m:<16}{"running":<10}{len(h)}/{trials_cfg[m]:<12}{"-":<10}-")
    else:
        print(f"{m:<16}{"pending":<10}{str(trials_cfg[m]):<12}{"-":<10}-")


In [ ]:
# Stage: train — 4 Results (checks outputs/manifest_index.json#train)
# Usa Stage.is_done() para skip si ya completado; Stage.run() publica manifest a manifest_index.json
!python main.py --stage train  # writes outputs/manifest_index.json#train.manifest
# Re-run (ignores is_done, no cascade): !python main.py --stage train --force
# Manifest: outputs/manifest_index.json (CSV+JSON, no XLSX) + tables/*.csv


In [ ]:
# (Opcional) Ejecutar pipeline completo — respeta is_done() por stage
# !python main.py --stage all  # orquestador declarativo, no recomputa stages completadas
print("usa celdas modulares por stage para trazabilidad; 'all' solo si buscas ejecución completa")


In [ ]:
# Verificar outputs en Drive (OUTPUT_DIR vía env, no absoluto) — manifest_index.json es single source of truth
import os, json, pathlib
from dotenv import load_dotenv
# Colab env comes from the mount cell above, never from .env files (local-only)
# (no load_dotenv in Colab)
OUTPUT_DIR = os.getenv("OUTPUT_DIR", "../outputs")
GDRIVE_NOTEBOOK_NAME = os.getenv("GDRIVE_NOTEBOOK_NAME", "experiments.ipynb")
manifest_path = pathlib.Path(OUTPUT_DIR) / "manifest_index.json"
print(f"[info] {manifest_path} (GDRIVE_NOTEBOOK_NAME={GDRIVE_NOTEBOOK_NAME})")
if manifest_path.exists():
    data = json.loads(manifest_path.read_text(encoding="utf-8"))
    print(json.dumps(data, indent=2, ensure_ascii=False)[:4000])
    if isinstance(data, dict):  # canonical index shape {stage: {seccion_paper, output_dir, manifest}}
        for stage, entry in data.items():
            manifest = entry.get('manifest', {}) if isinstance(entry, dict) else {}
            keys = list(manifest.keys())[:8] if isinstance(manifest, dict) else []
            print(f"- {stage}: {entry.get('seccion_paper')} -> {entry.get('output_dir')} keys={keys}")
    else:  # legacy list shape [{stage, seccion_paper, path, keys_principales}]
        for entry in data:
            print(f"- {entry.get('stage')}: {entry.get('seccion_paper')} -> {entry.get('path')} keys={entry.get('keys_principales')}")
else:
    print("no manifest yet — ejecuta stages arriba")
for p in sorted(pathlib.Path(OUTPUT_DIR).glob("tables/*")):
    print(p, p.stat().st_size)


In [ ]:
# Revisar logs y checkpoints (MCP gdrive readable: outputs/logs/)
import pathlib, os, json, glob
OUTPUT_DIR = os.getenv("OUTPUT_DIR", "../outputs")
logs_dir = pathlib.Path(OUTPUT_DIR) / "logs"
ckpt_dir = pathlib.Path(OUTPUT_DIR) / "checkpoints"
for pattern in [str(logs_dir / "errors.json"), str(logs_dir / "run_*.log"), str(ckpt_dir / "checkpoint_manifest.json")]:
    for f in glob.glob(pattern):
        fp = pathlib.Path(f)
        print(f"\n--- {fp} ({fp.stat().st_size} bytes) ---")
        print(fp.read_text(encoding="utf-8", errors="ignore")[:2000])
if not any(logs_dir.glob("*")):
    print("no logs yet")
if not any(ckpt_dir.glob("*")):
    print("no checkpoints yet")


In [ ]:
# Quality gates (config.yaml:quality_gates) — nunca bajar thresholds para pasar
import yaml, pathlib, json, os
OUTPUT_DIR = os.getenv("OUTPUT_DIR", "../outputs")
cfg = yaml.safe_load(open("config.yaml", encoding="utf-8"))
print(yaml.dump(cfg.get("quality_gates"), allow_unicode=True))
print("\nFlujo al fallar gate: fail -> diagnostic_log.md -> fix train.py -> re-run -> verify")
tm = pathlib.Path(OUTPUT_DIR) / "tables" / "train_manifest.json"
if tm.exists():
    m = json.loads(tm.read_text(encoding="utf-8"))
    print(json.dumps(m.get("metrics", {}), indent=2, ensure_ascii=False)[:3000])
